# 02 — GlotLID: zero-shot, target-only, rehearsal
Uses **GlotLID v3**, with the exact downloaded revision recorded. All original vocabulary rows, hash buckets, and output labels are retained.

Run notebook 00 first. Keep the same data and settings for every model. Checkpoints are large; run one model at a time.

Table 2 measures whether forgetting occurs. Table 3 measures mitigation. Neither outcome is assumed.

In [1]:
from pathlib import Path
import os, sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/lid_finetuning_bundle')]
ROOT = next((p for p in candidates if (p/'config.json').exists() and (p/'lidlab').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the complete ZIP first and set ROOT to its lid_finetuning_bundle folder.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project folder:', ROOT)


Project folder: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method


In [2]:
from lidlab.data import load_config
from lidlab.experiment import run_experiment
c = load_config()
MODEL = 'glotlid'
print('Starting checkpoint:', c['models'][MODEL])
print('Replay starts from:', c['replay_start'])

Starting checkpoint: {'repo_id': 'cis-lmu/glotlid', 'filename': 'model_v3.bin', 'revision': None, 'local_path': None, 'lr': 0.05, 'existing_zero_predictions': None}
Replay starts from: base


## Run all three conditions
The first run downloads and locks the original checkpoint. Both trained conditions keep every original label and add missing target labels. An extra untrained `initialized` evaluation measures the effect of adding labels alone.

Existing zero-shot predictions are imported only if configured and aligned with the evaluation index. Otherwise zero-shot is recalculated. Completed unchanged stages are reused. Interrupted stages restart from their defined initial checkpoint.

If you change data or hyperparameters, use a new `output_dir` before rerunning.

In [3]:
results = run_experiment(c, MODEL)
for phase, benchmark_results in results.items():
    for benchmark, frame in benchmark_results.items():
        print(phase, benchmark)
        display(frame[['language','precision','recall','f1','support']])

wili-2018.jsonl: NOTE ['arb_Arab'] absent by declaration; scored over 10 categories
glotlid zero_shot {'commonlid': 0.7445, 'flores_plus': 0.7687, 'wili_2018': 0.7303}
glotlid initialized {'commonlid': 0.7445, 'flores_plus': 0.7687, 'wili_2018': 0.7303}
glotlid target_only {'commonlid': 0.9622, 'flores_plus': 0.9896, 'wili_2018': 0.9727}
glotlid replay {'commonlid': 0.9694, 'flores_plus': 0.9954, 'wili_2018': 0.9775}
zero_shot commonlid


,language,precision,recall,f1,support
0,sin_Sinh,0.377963,0.976977,0.545059,2693
1,pli_Sinh,0.000000,0.000000,0.000000,3027
2,san_Sinh,0.000000,0.000000,0.000000,1333
3,san_Deva,0.987104,0.947132,0.966705,889
4,eng_Latn,0.992214,0.868292,0.926126,27447
5,tam_Taml,0.975904,1.000000,0.987805,81
6,hin_Deva,0.988421,0.954980,0.971413,3665
7,ben_Beng,1.000000,0.958621,0.978873,1885
8,arb_Arab,0.999835,0.932082,0.964771,26061
9,fra_Latn,0.983333,0.876238,0.926702,3232


zero_shot flores_plus


,language,precision,recall,f1,support
0,sin_Sinh,0.377963,0.976977,0.545059,2693
1,pli_Sinh,0.000000,0.000000,0.000000,3027
2,san_Sinh,0.000000,0.000000,0.000000,1327
3,san_Deva,1.000000,0.989130,0.994536,1012
4,eng_Latn,1.000000,1.000000,1.000000,1012
5,tam_Taml,1.000000,1.000000,1.000000,1012
6,hin_Deva,0.987280,0.997036,0.992134,1012
7,ben_Beng,1.000000,0.999012,0.999506,1012
8,arb_Arab,0.998864,0.860078,0.924290,1022
9,fra_Latn,1.000000,1.000000,1.000000,1012


zero_shot wili_2018


...

,language,precision,recall,f1,support
0,sin_Sinh,0.989090,0.976235,0.982620,2693
1,pli_Sinh,0.997670,0.990089,0.993865,3027
2,san_Sinh,0.996965,0.985746,0.991324,1333
3,san_Deva,0.921444,0.976378,0.948116,889
4,eng_Latn,0.970883,0.952454,0.961580,27447
5,tam_Taml,0.975904,1.000000,0.987805,81
6,hin_Deva,0.992312,0.950887,0.971158,3665
7,ben_Beng,1.000000,0.964987,0.982181,1885
8,arb_Arab,0.999457,0.988642,0.994020,26061
9,fra_Latn,0.986325,0.892636,0.937145,3232


replay flores_plus


,language,precision,recall,f1,support
0,sin_Sinh,0.989090,0.976235,0.982620,2693
1,pli_Sinh,0.997670,0.990089,0.993865,3027
2,san_Sinh,0.996965,0.990203,0.993573,1327
3,san_Deva,1.000000,1.000000,1.000000,1012
4,eng_Latn,0.981571,1.000000,0.990700,1012
5,tam_Taml,1.000000,1.000000,1.000000,1012
6,hin_Deva,0.996063,1.000000,0.998028,1012
7,ben_Beng,1.000000,0.999012,0.999506,1012
8,arb_Arab,0.997030,0.985323,0.991142,1022
9,fra_Latn,1.000000,0.999012,0.999506,1012


replay wili_2018


,language,precision,recall,f1,support
0,sin_Sinh,0.989090,0.976235,0.982620,2693
1,pli_Sinh,0.997670,0.990089,0.993865,3027
2,san_Sinh,0.996965,0.980597,0.988713,1340
3,san_Deva,0.998982,0.997965,0.998473,983
4,eng_Latn,0.856164,1.000000,0.922509,1000
5,tam_Taml,1.000000,0.989827,0.994888,983
6,hin_Deva,0.995910,0.974975,0.985331,999
7,ben_Beng,1.000000,0.895000,0.944591,1000
8,arb_Arab,0.000000,0.000000,NaN,0
9,fra_Latn,0.984000,0.991935,0.987952,992


## Inspect forgetting and recovery

In [4]:
import pandas as pd
for benchmark in results['zero_shot']:
    base = results['zero_shot'][benchmark].set_index('language')
    target = results['target_only'][benchmark].set_index('language')
    replay = results['replay'][benchmark].set_index('language')
    comparison = pd.DataFrame({'zero_shot_f1':base.f1, 'target_only_f1':target.f1,
                               'replay_f1':replay.f1, 'drop_after_target_only':base.f1-target.f1,
                               'improvement_with_replay':replay.f1-target.f1})
    print(benchmark)
    display(comparison)
print('Scores are on a 0–1 scale. Positive drop = forgetting; negative drop = improvement.')

commonlid


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.545059,0.975875,0.982620,-0.430816,0.006745
pli_Sinh,0.000000,0.988982,0.993865,-0.988982,0.004883
san_Sinh,0.000000,0.992084,0.991324,-0.992084,-0.000760
san_Deva,0.966705,0.964490,0.948116,0.002215,-0.016374
eng_Latn,0.926126,0.929356,0.961580,-0.003230,0.032224
tam_Taml,0.987805,0.981818,0.987805,0.005987,0.005987
hin_Deva,0.971413,0.961755,0.971158,0.009658,0.009403
ben_Beng,0.978873,0.980530,0.982181,-0.001657,0.001651
arb_Arab,0.964771,0.961032,0.994020,0.003739,0.032988


flores_plus


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.545059,0.995559,0.982620,-0.450500,-0.012939
pli_Sinh,0.000000,0.995695,0.993865,-0.995695,-0.001830
san_Sinh,0.000000,0.994333,0.993573,-0.994333,-0.000760
san_Deva,0.994536,0.995533,1.000000,-0.000998,0.004467
eng_Latn,1.000000,1.000000,0.990700,0.000000,-0.009300
tam_Taml,1.000000,1.000000,1.000000,0.000000,0.000000
hin_Deva,0.992134,0.993103,0.998028,-0.000970,0.004924
ben_Beng,0.999506,0.999506,0.999506,0.000000,0.000000
arb_Arab,0.924290,0.912187,0.991142,0.012103,0.078954


wili_2018


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.545059,0.995559,0.982620,-0.450500,-0.012939
pli_Sinh,0.000000,0.995695,0.993865,-0.995695,-0.001830
san_Sinh,0.000000,0.989474,0.988713,-0.989474,-0.000760
san_Deva,0.997453,0.997453,0.998473,0.000000,0.001020
eng_Latn,0.938563,0.937618,0.922509,0.000945,-0.015109
tam_Taml,0.994888,0.994888,0.994888,0.000000,0.000000
hin_Deva,0.918422,0.910675,0.985331,0.007747,0.074656
ben_Beng,0.945148,0.945148,0.944591,0.000000,-0.000557
arb_Arab,NaN,NaN,NaN,NaN,NaN


Scores are on a 0–1 scale. Positive drop = forgetting; negative drop = improvement.


After all three model notebooks finish, run notebook 04. Do not infer preservation of all original languages from eight replay-language scores. The original label set remains available, but retention needs evaluation.